# Solutions — Loops, Control Flow & Log Processing

Exercise notebook: `01_python_fundamentals/03_loops_and_log_processing.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
SAMPLE_LOG = LAB / "datasets/logs/app_small.log"   # 250 real-looking lines
lines = SAMPLE_LOG.read_text(encoding="utf-8").splitlines()
print(len(lines), "lines; first:", lines[0][:90])

## Exercise 1 — 🟢 Easy · Count log levels

**Working solution**

In [ ]:
def count_levels(lines):
    counts = {}
    for line in lines:
        parts = line.split()
        if len(parts) < 2 or not parts[0][:4].isdigit():
            continue
        level = parts[1]
        counts[level] = counts.get(level, 0) + 1
    return counts

**Explanation**

Guard clause + `continue` keeps the happy path unindented. `counts.get(k, 0) + 1` is the plain-dict counter idiom (you'll meet `Counter` in Level 2).

**Complexity:** O(n).

**Common mistakes**

- `for i in range(len(lines)):` — iterate the list directly.

## Exercise 2 — 🟡 Medium · Find slow requests

**Working solution**

In [ ]:
def find_slow_requests(lines, threshold_ms):
    slow = []
    for line in lines:
        fields = dict(tok.split("=", 1) for tok in line.split() if "=" in tok)
        req, latency = fields.get("req"), fields.get("latency_ms")
        if req is None or latency is None or not latency.isdigit():
            continue
        if int(latency) > threshold_ms:
            slow.append((req, int(latency)))
    return slow

**Explanation**

Building a small dict per line from `key=value` tokens makes the lookups order-independent — the log format can add fields without breaking you.

**Complexity:** O(total tokens).

**Common mistakes**

- Indexing `line.split()[5]` — the next deploy adds a field and everything shifts.

## Exercise 3 — 🟡 Medium · Merge multi-line log entries

**Working solution**

In [ ]:
import re

ENTRY_START = re.compile(r"^\d{4}-\d{2}-\d{2}T")

def merge_multiline(lines):
    entries = []
    for line in lines:
        if not line.strip():
            continue
        if ENTRY_START.match(line):
            entries.append(line)
        elif entries:
            entries[-1] += "\n" + line
    return entries

**Explanation**

`entries[-1]` is the current entry — no separate `current` variable or flush-at-end bug to forget.

**Complexity:** O(n) amortized (string concatenation per continuation is fine at this scale; collect lists for huge entries).

**Common mistakes**

- Keeping a `current` buffer and forgetting to append it after the loop ends.

## Exercise 4 — 🟠 Hard · Sessionize user activity

**Working solution**

In [ ]:
from datetime import datetime, timedelta

def count_sessions(events, gap_minutes=30):
    gap = timedelta(minutes=gap_minutes)
    last_seen = {}
    sessions = {}
    for user, ts in sorted(events, key=lambda e: e[1]):
        previous = last_seen.get(user)
        if previous is None or ts - previous > gap:
            sessions[user] = sessions.get(user, 0) + 1
        last_seen[user] = ts
    return sessions

**Explanation**

Sort once by timestamp, then keep per-user state in dicts. Tuple unpacking in the `for` header
(`for user, ts in ...`) replaces `event.getUser()` / `event.getTs()`. `datetime - datetime` gives a `timedelta`.

**Complexity:** O(n log n) for the sort, O(n) for the scan.

**Common mistakes**

- Assuming input is sorted.
- Using `>=` instead of `>` — exactly 30 minutes is the same session per the spec.

## Challenge — 🔴 Very Hard · Error-burst alerting with a sliding window

**Working solution**

In [ ]:
from collections import deque
from datetime import datetime, timedelta

def error_bursts(entries, window_seconds=60, threshold=5):
    window = deque()
    alerts = []
    span = timedelta(seconds=window_seconds)
    for ts, level in entries:
        if level not in ("ERROR", "CRITICAL"):
            continue
        window.append(ts)
        while window and window[0] <= ts - span:
            window.popleft()
        if len(window) >= threshold:
            alerts.append(ts)
            window.clear()
    return alerts

**Explanation**

A deque gives O(1) appends/pops at both ends — the standard sliding-window structure. Each error enters and
leaves the deque once, so the whole scan is O(n).

**Complexity:** O(n) time, O(threshold) space.

**Common mistakes**

- Re-counting the whole list for every entry (O(n²)).
- Off-by-one on the window edge: the spec says `(ts - w, ts]`, so an entry exactly `w` seconds old is out.